# Setup

In [1]:
from pathlib import Path
import pickle
import random

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

PROJECT_ROOT = Path.cwd().resolve().parent
TRAJ_PATH = PROJECT_ROOT / "data" / "topic_trajectories.pkl"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("TRAJ_PATH exists:", TRAJ_PATH.exists(), TRAJ_PATH)
print("Device:", device)

TRAJ_PATH exists: True /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation/data/topic_trajectories.pkl
Device: cpu


In [2]:
# Load trajectories
with open(TRAJ_PATH, "rb") as f:
    topic_trajectories = pickle.load(f)

print("Number of topics loaded:", len(topic_trajectories))

for topic_id, info in list(topic_trajectories.items())[:3]:
    print(f"\nTopic {topic_id}")
    print("Label:", info["label"])
    print("Years:", info["years"])
    print("Trajectory shape:", info["trajectory"].shape)

Number of topics loaded: 15

Topic 0
Label: Public Health & Workforce Impact
Years: [2020, 2021, 2022]
Trajectory shape: (3, 384)

Topic 1
Label: Online Education & Training
Years: [2020, 2021, 2022]
Trajectory shape: (3, 384)

Topic 2
Label: Clinical Virology & Infection
Years: [2020, 2021, 2022]
Trajectory shape: (3, 384)


In [3]:
# build transition dataset
transition_rows = []

for topic_id, info in topic_trajectories.items():
    years = info["years"]
    traj = info["trajectory"]
    label = info["label"]

    for t in range(len(years) - 1):
        transition_rows.append({
            "topic_id": topic_id,
            "topic_label": label,
            "year_from": years[t],
            "year_to": years[t + 1],
            "x_t": traj[t].astype(np.float32),
            "x_t1": traj[t + 1].astype(np.float32),
        })

transitions_df = pd.DataFrame(transition_rows)

print("Transition dataset shape:", transitions_df.shape)
display(transitions_df.head())

Transition dataset shape: (27, 6)


,topic_id,topic_label,year_from,year_to,x_t,x_t1
0,0,Public Health & Workforce Impact,2020,2021,"[0.022808092, 0.03738438, -0.025412643, 0.0275...","[0.00894767, 0.015065819, -0.009621622, 0.0234..."
1,0,Public Health & Workforce Impact,2021,2022,"[0.00894767, 0.015065819, -0.009621622, 0.0234...","[0.036359653, 0.0104295965, 0.013622075, 0.005..."
2,1,Online Education & Training,2020,2021,"[0.01232472, 0.029829707, 0.0024571244, 0.0058...","[0.008245414, 0.015392465, 0.0030519473, -0.02..."
3,1,Online Education & Training,2021,2022,"[0.008245414, 0.015392465, 0.0030519473, -0.02...","[0.022293752, 0.006708989, -0.041292395, -0.02..."
4,2,Clinical Virology & Infection,2020,2021,"[0.010815447, 0.026126163, -0.027783157, 0.007...","[-0.007693753, 0.017500222, -0.029099848, 0.00..."


In [4]:
# stack transition matrices
X_t = np.vstack(transitions_df["x_t"].values).astype(np.float32)
X_t1 = np.vstack(transitions_df["x_t1"].values).astype(np.float32)

print("X_t shape:", X_t.shape)
print("X_t1 shape:", X_t1.shape)

X_t shape: (27, 384)
X_t1 shape: (27, 384)


# Noise Schedule

In [5]:
# Define noise levels
noise_levels = np.array([0.01, 0.03, 0.05, 0.08, 0.12], dtype=np.float32)

print("Noise levels:")
print(noise_levels)

Noise levels:
[0.01 0.03 0.05 0.08 0.12]


In [6]:
# Expand transitions
rng = np.random.default_rng(SEED)

expanded_rows = []

for i in range(len(X_t)):
    x_t = X_t[i]
    x_t1 = X_t1[i]
    topic_id = transitions_df.iloc[i]["topic_id"]
    topic_label = transitions_df.iloc[i]["topic_label"]
    year_from = transitions_df.iloc[i]["year_from"]
    year_to = transitions_df.iloc[i]["year_to"]

    for sigma in noise_levels:
        noise = rng.normal(loc=0.0, scale=float(sigma), size=x_t1.shape).astype(np.float32)
        x_t1_noisy = x_t1 + noise

        expanded_rows.append({
            "topic_id": topic_id,
            "topic_label": topic_label,
            "year_from": year_from,
            "year_to": year_to,
            "sigma": float(sigma),
            "x_t": x_t,
            "x_t1_clean": x_t1,
            "x_t1_noisy": x_t1_noisy,
            "noise": noise,
        })

expanded_df = pd.DataFrame(expanded_rows)

print("Expanded training set shape:", expanded_df.shape)
display(expanded_df.head())

Expanded training set shape: (135, 9)


,topic_id,topic_label,year_from,year_to,sigma,x_t,x_t1_clean,x_t1_noisy,noise
0,0,Public Health & Workforce Impact,2020,2021,0.01,"[0.022808092, 0.03738438, -0.025412643, 0.0275...","[0.00894767, 0.015065819, -0.009621622, 0.0234...","[0.0119948415, 0.0046659783, -0.0021171104, 0....","[0.0030471708, -0.010399841, 0.0075045116, 0.0..."
1,0,Public Health & Workforce Impact,2020,2021,0.03,"[0.022808092, 0.03738438, -0.025412643, 0.0275...","[0.00894767, 0.015065819, -0.009621622, 0.0234...","[-0.039090667, -0.008758269, 0.0035674758, 0.0...","[-0.048038337, -0.023824088, 0.013189098, 0.01..."
2,0,Public Health & Workforce Impact,2020,2021,0.05,"[0.022808092, 0.03738438, -0.025412643, 0.0275...","[0.00894767, 0.015065819, -0.009621622, 0.0234...","[-0.055044375, 0.09915665, 0.076828144, 0.0914...","[-0.063992046, 0.08409083, 0.086449765, 0.0679..."
3,0,Public Health & Workforce Impact,2020,2021,0.08,"[0.022808092, 0.03738438, -0.025412643, 0.0275...","[0.00894767, 0.015065819, -0.009621622, 0.0234...","[0.030759292, -0.029852822, 0.046203725, 0.032...","[0.021811621, -0.04491864, 0.055825345, 0.0088..."
4,0,Public Health & Workforce Impact,2020,2021,0.12,"[0.022808092, 0.03738438, -0.025412643, 0.0275...","[0.00894767, 0.015065819, -0.009621622, 0.0234...","[0.09885442, -0.1874159, -0.12125338, 0.205416...","[0.08990675, -0.20248172, -0.11163176, 0.18196..."


In [7]:
# stack arrays
X_curr = np.vstack(expanded_df["x_t"].values).astype(np.float32)
X_next_clean = np.vstack(expanded_df["x_t1_clean"].values).astype(np.float32)
X_next_noisy = np.vstack(expanded_df["x_t1_noisy"].values).astype(np.float32)
X_noise = np.vstack(expanded_df["noise"].values).astype(np.float32)
sigmas = expanded_df["sigma"].to_numpy(dtype=np.float32).reshape(-1, 1)

print("X_curr shape:", X_curr.shape)
print("X_next_clean shape:", X_next_clean.shape)
print("X_next_noisy shape:", X_next_noisy.shape)
print("X_noise shape:", X_noise.shape)
print("sigmas shape:", sigmas.shape)

X_curr shape: (135, 384)
X_next_clean shape: (135, 384)
X_next_noisy shape: (135, 384)
X_noise shape: (135, 384)
sigmas shape: (135, 1)


# Mini Denoiser MLP

In [8]:
#TV Split
from sklearn.model_selection import train_test_split

idx = np.arange(len(X_curr))

train_idx, val_idx = train_test_split(
    idx,
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)

print("Train size:", len(train_idx))
print("Val size:", len(val_idx))

Train size: 108
Val size: 27


In [9]:
# Dataset class
class TopicDiffusionDataset(Dataset):
    def __init__(self, X_curr, X_next_noisy, sigmas, X_next_clean):
        self.X_curr = torch.tensor(X_curr, dtype=torch.float32)
        self.X_next_noisy = torch.tensor(X_next_noisy, dtype=torch.float32)
        self.sigmas = torch.tensor(sigmas, dtype=torch.float32)
        self.X_next_clean = torch.tensor(X_next_clean, dtype=torch.float32)

    def __len__(self):
        return len(self.X_curr)

    def __getitem__(self, idx):
        return {
            "x_curr": self.X_curr[idx],
            "x_next_noisy": self.X_next_noisy[idx],
            "sigma": self.sigmas[idx],
            "x_next_clean": self.X_next_clean[idx],
        }

In [10]:
# Create loaders
train_dataset = TopicDiffusionDataset(
    X_curr[train_idx],
    X_next_noisy[train_idx],
    sigmas[train_idx],
    X_next_clean[train_idx]
)

val_dataset = TopicDiffusionDataset(
    X_curr[val_idx],
    X_next_noisy[val_idx],
    sigmas[val_idx],
    X_next_clean[val_idx]
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Train batches: 7
Val batches: 2


In [11]:
# Define Model
class DenoiserMLP(nn.Module):
    def __init__(self, dim=384, hidden_dim=512):
        super().__init__()

        input_dim = dim + dim + 1  # x_curr + x_next_noisy + sigma

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, dim),
        )

    def forward(self, x_curr, x_next_noisy, sigma):
        x = torch.cat([x_curr, x_next_noisy, sigma], dim=1)
        return self.net(x)

In [12]:
#Initialize Model
model = DenoiserMLP(dim=384, hidden_dim=512).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)

DenoiserMLP(
  (net): Sequential(
    (0): Linear(in_features=769, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=384, bias=True)
  )
)


In [13]:
# Training loop
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_n = 0

    for batch in loader:
        x_curr = batch["x_curr"].to(device)
        x_next_noisy = batch["x_next_noisy"].to(device)
        sigma = batch["sigma"].to(device)
        x_next_clean = batch["x_next_clean"].to(device)

        with torch.set_grad_enabled(is_train):
            pred = model(x_curr, x_next_noisy, sigma)
            loss = criterion(pred, x_next_clean)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        batch_size = x_curr.size(0)
        total_loss += loss.item() * batch_size
        total_n += batch_size

    return total_loss / total_n

In [14]:
EPOCHS = 100

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
}

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer=optimizer)
    val_loss = run_epoch(model, val_loader, optimizer=None)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

Epoch   1 | train_loss=0.001499 | val_loss=0.000932
Epoch  10 | train_loss=0.000075 | val_loss=0.000171
Epoch  20 | train_loss=0.000011 | val_loss=0.000077
Epoch  30 | train_loss=0.000008 | val_loss=0.000063
Epoch  40 | train_loss=0.000012 | val_loss=0.000062
Epoch  50 | train_loss=0.000009 | val_loss=0.000066
Epoch  60 | train_loss=0.000008 | val_loss=0.000066
Epoch  70 | train_loss=0.000003 | val_loss=0.000057
Epoch  80 | train_loss=0.000011 | val_loss=0.000064
Epoch  90 | train_loss=0.000007 | val_loss=0.000058
Epoch 100 | train_loss=0.000004 | val_loss=0.000058


In [15]:
history_df = pd.DataFrame(history)
display(history_df.tail(10))

,epoch,train_loss,val_loss
90,91,0.000008,0.000059
91,92,0.000006,0.000061
92,93,0.000006,0.000057
93,94,0.000005,0.000055
94,95,0.000005,0.000061
95,96,0.000005,0.000054
96,97,0.000005,0.000060
97,98,0.000005,0.000056
98,99,0.000007,0.000057
99,100,0.000004,0.000058


In [16]:
# denoiser vs noisy baseline
model.eval()

val_preds = []
val_targets = []
val_noisy = []

with torch.no_grad():
    for batch in val_loader:
        x_curr = batch["x_curr"].to(device)
        x_next_noisy = batch["x_next_noisy"].to(device)
        sigma = batch["sigma"].to(device)
        x_next_clean = batch["x_next_clean"].to(device)

        pred = model(x_curr, x_next_noisy, sigma)

        val_preds.append(pred.cpu().numpy())
        val_targets.append(x_next_clean.cpu().numpy())
        val_noisy.append(x_next_noisy.cpu().numpy())

Y_val_pred = np.vstack(val_preds)
Y_val_true = np.vstack(val_targets)
Y_val_noisy = np.vstack(val_noisy)

mse_noisy = np.mean((Y_val_noisy - Y_val_true) ** 2)
mse_pred = np.mean((Y_val_pred - Y_val_true) ** 2)

print("Validation MSE of noisy next-state vs clean next-state:", mse_noisy)
print("Validation MSE of denoised prediction vs clean next-state:", mse_pred)
print("Improvement:", mse_noisy - mse_pred)

Validation MSE of noisy next-state vs clean next-state: 0.0040862486
Validation MSE of denoised prediction vs clean next-state: 5.8128862e-05
Improvement: 0.0040281196


In [17]:
# Get latest topic states
latest_states = []

for topic_id, info in topic_trajectories.items():
    years = info["years"]
    traj = info["trajectory"]

    if len(years) > 0:
        latest_states.append({
            "topic_id": topic_id,
            "topic_label": info["label"],
            "latest_year": years[-1],
            "x_latest": traj[-1].astype(np.float32),
        })

latest_df = pd.DataFrame(latest_states)

print("Latest states shape:", latest_df.shape)
display(latest_df.head())

Latest states shape: (15, 4)


,topic_id,topic_label,latest_year,x_latest
0,0,Public Health & Workforce Impact,2022,"[0.036359653, 0.0104295965, 0.013622075, 0.005..."
1,1,Online Education & Training,2022,"[0.022293752, 0.006708989, -0.041292395, -0.02..."
2,2,Clinical Virology & Infection,2022,"[-0.012340484, 0.01592598, -0.019865632, 0.004..."
3,3,Medical Imaging & Deep Learning,2022,"[-0.0033894896, 0.020917473, 0.030582165, -0.0..."
4,4,Financial Markets & Economic Impact,2022,"[-0.057611395, -0.09756207, 0.04446043, 0.1054..."


In [18]:
# only 2022
future_df = latest_df[latest_df["latest_year"] == 2022].copy()

X_latest = np.vstack(future_df["x_latest"].values).astype(np.float32)

print("Topics eligible for 2023 simulation:", len(future_df))
display(future_df[["topic_id", "topic_label", "latest_year"]])

Topics eligible for 2023 simulation: 12


,topic_id,topic_label,latest_year
0,0,Public Health & Workforce Impact,2022
1,1,Online Education & Training,2022
2,2,Clinical Virology & Infection,2022
3,3,Medical Imaging & Deep Learning,2022
4,4,Financial Markets & Economic Impact,2022
5,5,Mental Health & Psychological Effects,2022
8,8,Severe Disease & Immune Response,2022
9,9,Food Systems & Hospitality Impact,2022
10,10,"Prevention, Masks & Environmental Effects",2022
11,11,"Policy, Governance & Public Response",2022


In [19]:
# future proposals
FUTURE_SIGMA = 0.08

rng = np.random.default_rng(SEED)
X_future_noisy = X_latest + rng.normal(
    loc=0.0,
    scale=FUTURE_SIGMA,
    size=X_latest.shape
).astype(np.float32)

sigma_future = np.full((len(X_latest), 1), FUTURE_SIGMA, dtype=np.float32)

print("X_future_noisy shape:", X_future_noisy.shape)
print("sigma_future shape:", sigma_future.shape)

X_future_noisy shape: (12, 384)
sigma_future shape: (12, 1)


In [20]:
# denoising
model.eval()

with torch.no_grad():
    x_curr_t = torch.tensor(X_latest, dtype=torch.float32, device=device)
    x_next_noisy_t = torch.tensor(X_future_noisy, dtype=torch.float32, device=device)
    sigma_t = torch.tensor(sigma_future, dtype=torch.float32, device=device)

    X_future_pred = model(x_curr_t, x_next_noisy_t, sigma_t).cpu().numpy()

print("Predicted future shape:", X_future_pred.shape)

Predicted future shape: (12, 384)


In [21]:
# measure future movement
movement_norm = np.linalg.norm(X_future_pred - X_latest, axis=1)

future_results_df = future_df[["topic_id", "topic_label", "latest_year"]].copy()
future_results_df["future_sigma"] = FUTURE_SIGMA
future_results_df["movement_norm"] = movement_norm

display(future_results_df.sort_values("movement_norm", ascending=False))

,topic_id,topic_label,latest_year,future_sigma,movement_norm
9,9,Food Systems & Hospitality Impact,2022,0.08,0.339089
0,0,Public Health & Workforce Impact,2022,0.08,0.298353
11,11,"Policy, Governance & Public Response",2022,0.08,0.287237
4,4,Financial Markets & Economic Impact,2022,0.08,0.264548
5,5,Mental Health & Psychological Effects,2022,0.08,0.211100
10,10,"Prevention, Masks & Environmental Effects",2022,0.08,0.210937
14,14,Hospitalization & Patient Outcomes,2022,0.08,0.184577
1,1,Online Education & Training,2022,0.08,0.164708
12,12,Clinical Care & Procedures,2022,0.08,0.112117
2,2,Clinical Virology & Infection,2022,0.08,0.099076


In [22]:
# latest vs predicted
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

future_results_df["cosine_similarity_to_latest"] = [
    cosine_sim(X_future_pred[i], X_latest[i])
    for i in range(len(X_latest))
]

display(
    future_results_df.sort_values("cosine_similarity_to_latest")
)

,topic_id,topic_label,latest_year,future_sigma,movement_norm,cosine_similarity_to_latest
0,0,Public Health & Workforce Impact,2022,0.08,0.298353,0.926739
11,11,"Policy, Governance & Public Response",2022,0.08,0.287237,0.939108
9,9,Food Systems & Hospitality Impact,2022,0.08,0.339089,0.955670
5,5,Mental Health & Psychological Effects,2022,0.08,0.211100,0.971219
4,4,Financial Markets & Economic Impact,2022,0.08,0.264548,0.971807
14,14,Hospitalization & Patient Outcomes,2022,0.08,0.184577,0.977872
10,10,"Prevention, Masks & Environmental Effects",2022,0.08,0.210937,0.986226
1,1,Online Education & Training,2022,0.08,0.164708,0.986498
12,12,Clinical Care & Procedures,2022,0.08,0.112117,0.991714
2,2,Clinical Virology & Infection,2022,0.08,0.099076,0.995004


In [ ]:
# TODO
# - compare clean-state prediction vs noise-prediction objective
# - try a train/validation split by topic rather than random transition rows
# - test multiple future sigma values
# - try residual MLP or deeper network
# - compare against persistence baseline: x_{t+1} = x_t